# setting up cicd pipelines

## 📚 Learning Objectives

By completing this notebook, you will:
- Train a baseline classifier with scikit-learn
- Evaluate using accuracy + confusion matrix
- Show how to encode categorical features

## 🔗 Prerequisites

- ✅ Python basics
- ✅ Jupyter Notebook basics

---

## Official Structure Reference

This notebook covers practical activities from **Course 11, Unit 4**:
- setting up cicd pipelines
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md`

---


## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# small synthetic dataset
rng = np.random.default_rng(123)
n = 600
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
color = rng.choice(['red','green','blue'], size=n)

y = ((x1 + 0.8*x2 + (color == 'red')*0.6 + rng.normal(scale=0.5, size=n)) > 0.2).astype(int)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'color': color, 'y': y})
X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pre = ColumnTransformer([
 ('cat', OneHotEncoder(handle_unknown='ignore'), ['color']),
], remainder='passthrough')

clf = Pipeline([
 ('pre', pre),
 ('model', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred))
print('confusion matrix:', confusion_matrix(y_test, y_pred))
print('\nreport:', classification_report(y_test, y_pred))


## 🌍 Real-World Application: CI/CD Pipeline for ML Models

Every production ML team uses automated pipelines to test and deploy models safely. Below is a working simulation of the CI/CD workflow used by companies like Airbnb and Spotify.

In [ ]:
import json, time, hashlib, pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("=== ML CI/CD Pipeline Simulation ===")
print("Simulating the pipeline used at Airbnb, Spotify, Netflix\n")

# ── Stage 1: Data validation ───────────────────────────────────────────────
def validate_data(X, y):
    assert X.shape[1] == 4, "Expected 4 features"
    assert len(np.unique(y)) == 3, "Expected 3 classes"
    assert not np.any(np.isnan(X)), "No NaN values allowed"
    print("  [PASS] Data validation: shape OK, no NaN, correct classes")
    return True

# ── Stage 2: Model training & evaluation ──────────────────────────────────
def train_and_evaluate(X_train, X_test, y_train, y_test, n_estimators=100):
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    return clf, acc

# ── Stage 3: Model quality gate ──────────────────────────────────────────
def quality_gate(accuracy, threshold=0.90):
    status = "PASS" if accuracy >= threshold else "FAIL"
    print(f"  [{status}] Quality gate: accuracy={accuracy:.3f} (threshold={threshold})")
    return status == "PASS"

# ── Stage 4: Model registry push ─────────────────────────────────────────
def register_model(clf, acc, version):
    model_hash = hashlib.md5(str(clf.get_params()).encode()).hexdigest()[:8]
    record = {'version': version, 'accuracy': round(acc,4), 'hash': model_hash, 'ts': time.time()}
    print(f"  [REGISTERED] v{version} — acc={acc:.4f}, hash={model_hash}")
    return record

# ── Run the full pipeline ─────────────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline_results = []
for run_id, n_est in enumerate([50, 100, 150], start=1):
    print(f"\n--- Pipeline Run #{run_id} (n_estimators={n_est}) ---")
    validate_data(X_train, y_train)
    clf, acc = train_and_evaluate(X_train, X_test, y_train, y_test, n_estimators=n_est)
    passed = quality_gate(acc)
    if passed:
        record = register_model(clf, acc, version=f"1.{run_id}.0")
        pipeline_results.append(record)
    else:
        print(f"  [BLOCKED] Model not deployed — accuracy below threshold")

# Plot pipeline results
fig, ax = plt.subplots(figsize=(8, 4))
versions = [r['version'] for r in pipeline_results]
accs     = [r['accuracy'] for r in pipeline_results]
ax.bar(versions, accs, color='steelblue')
ax.axhline(0.90, color='red', linestyle='--', label='Quality threshold')
ax.set_ylim([0.85, 1.01]); ax.set_ylabel("Accuracy"); ax.set_title("CI/CD Pipeline — Model Quality by Version")
ax.legend(); plt.tight_layout(); plt.savefig('/tmp/cicd_results.png', dpi=72)
print("\nCI/CD pipeline complete. Only models above threshold were registered.")
print("Real-world: GitHub Actions + MLflow is this exact workflow used at Airbnb.")

## 📝 Summary

You set up a **CI/CD pipeline** for ML models — automatically testing, validating, and deploying on every code push. MLOps = DevOps + ML-specific concerns (data drift, model decay, experiment tracking). Companies like Netflix and Google re-deploy models hundreds of times per day using these pipelines.